In [7]:
"""
Production-Grade Feature Extraction Pipeline
Paderborn Bearing Dataset (HH, IR, OR)

================================================================================
WINDOW SIZE ANALYSIS & RECOMMENDATION
================================================================================
Theoretical frequency-resolution requirements for the specified tolerances:
  - Envelope ±1.0 Hz  → requires Δf ≤ 2.0 Hz  → W ≥ 32,000 samples
  - Current ±1.5 Hz   → requires Δf ≤ 3.0 Hz  → W ≥ 21,334 samples
  - Current ±2.0 Hz   → requires Δf ≤ 4.0 Hz  → W ≥ 16,000 samples

W = 4096 (64 ms) yields ~124 windows/file but CANNOT resolve the specified
spectral tolerances (native Δf = 15.6 Hz; envelope 2× zero-pad Δf = 7.8 Hz).
Spectral peaks at BPFO/BPFI would fall between FFT bins, making the envelope
and sideband features physically imprecise.

RECOMMENDED: W = 8192, H = 4096
  • 61 windows per file (183 total) — sufficient for traditional ML
  • Envelope FFT with 2× zero-padding: Δf = 3.906 Hz
    → Uses PARABOLIC INTERPOLATION for sub-bin peak energy estimation
  • Welch PSD uses nperseg=4096 → 2 averaged segments per window (lower variance)
  • Captures ~3.1 shaft rotations per window (better cyclostationary stability)
  • Rotation alignment: 3.12 revs (deviation 0.12 from integer — good)

If you strictly need 125 windows, set W = 4096 and H = 2046 below, but be
aware that envelope / sideband features will be coarse approximations.
"""

from pathlib import Path
from typing import Dict, Tuple, List
import warnings

import numpy as np
import pandas as pd
from scipy import signal, stats
from scipy.signal import welch, hilbert, coherence

# =============================================================================
# CONFIGURATION — Physical Constants
# =============================================================================
FS: int = 64_000          # Sampling rate [Hz]
N: int = 256_000          # Trimmed samples per file
FE: float = 100.0         # Electrical harmonic of interest [Hz]
FR: float = 24.41         # Shaft frequency [Hz]

# Bearing fault frequencies (exact multipliers from prompt)
BPFO: float = 3.0530 * FR   # ~74.5237 Hz
BPFI: float = 4.9473 * FR   # ~120.7636 Hz
FTF: float  = 0.3818 * FR   # ~9.3197 Hz
BSF: float  = 1.9918 * FR   # ~48.6198 Hz

# Windowing parameters  ← ADJUST HERE
W: int = 8192             # Window size [samples]  ← RECOMMENDED
H: int = W // 2           # Hop size [samples] (50% overlap)

# Welch segment length (fixed per specification)
NPERSEG: int = 4096

# Labels mapping
LABEL_MAP: Dict[str, int] = {"HH": 0, "IR": 1, "OR": 2}
FILES: Dict[str, str] = {"HH": "/home/shawky/Documents/nti/actual nti/paderborn/Processed subset used in our STR-DDPM experiments/paderborn_subset_used_in_this_study/University of Paderborn Electric Motor Dataset/HH.csv", "IR": "/home/shawky/Documents/nti/actual nti/paderborn/Processed subset used in our STR-DDPM experiments/paderborn_subset_used_in_this_study/University of Paderborn Electric Motor Dataset/IR.csv", "OR": "/home/shawky/Documents/nti/actual nti/paderborn/Processed subset used in our STR-DDPM experiments/paderborn_subset_used_in_this_study/University of Paderborn Electric Motor Dataset/OR.csv"}

# =============================================================================
# INPUT VALIDATION
# =============================================================================

def _validate_1d(x: np.ndarray, expected_len: int, name: str) -> np.ndarray:
    """Validate that x is a 1-D numpy array of exact length."""
    if not isinstance(x, np.ndarray):
        raise TypeError(f"{name}: expected np.ndarray, got {type(x).__name__}")
    if x.ndim != 1:
        raise ValueError(f"{name}: expected 1-D array, got shape {x.shape}")
    if len(x) != expected_len:
        raise ValueError(f"{name}: expected length {expected_len}, got {len(x)}")
    return x


def _sum_magnitude_near(spectrum: np.ndarray, freqs: np.ndarray,
                        target: float, tol: float) -> float:
    """
    Sum magnitude values within ±tol Hz of target frequency.
    For coarse resolution (bin width > tol), this effectively returns
    the nearest bin(s) weighted by overlap with the tolerance band.
    """
    mask = np.abs(freqs - target) <= tol
    if not np.any(mask):
        return 0.0
    return float(np.sum(spectrum[mask]))


def _band_energy(spectrum: np.ndarray, freqs: np.ndarray,
                 f_low: float, f_high: float) -> float:
    """Integrate power spectrum over a frequency band via trapezoidal rule."""
    mask = (freqs >= f_low) & (freqs <= f_high)
    if not np.any(mask):
        return 0.0
    return float(np.trapz(spectrum[mask], freqs[mask]))


# =============================================================================
# VIBRATION — TIME DOMAIN
# =============================================================================

def extract_vib_time(x: np.ndarray) -> Dict[str, float]:
    """
    Extract time-domain vibration features.
    
    Parameters
    ----------
    x : np.ndarray
        Zero-mean vibration window of length W.
        
    Returns
    -------
    dict
        Dictionary with exact column names as keys.
    """
    x = _validate_1d(x, W, "vib_time")
    
    mean_val = float(x.mean())
    std_val = float(x.std(ddof=0))
    rms_val = float(np.sqrt(np.mean(x ** 2)))
    peak_val = float(np.max(np.abs(x)))
    p2p_val = float(np.ptp(x))
    skew_val = float(stats.skew(x, bias=False))
    kurt_val = float(stats.kurtosis(x, fisher=True, bias=False))
    
    crest = peak_val / (rms_val + 1e-20)
    abs_x = np.abs(x)
    mean_abs = float(np.mean(abs_x))
    mean_sqrt_abs = float(np.mean(np.sqrt(abs_x)))
    
    clearance = peak_val / (mean_sqrt_abs ** 2 + 1e-20)
    shape_factor = rms_val / (mean_abs + 1e-20)
    impulse_factor = peak_val / (mean_abs + 1e-20)
    
    return {
        "vib_mean": mean_val,
        "vib_std": std_val,
        "vib_rms": rms_val,
        "vib_peak": peak_val,
        "vib_p2p": p2p_val,
        "vib_skew": skew_val,
        "vib_kurt": kurt_val,
        "vib_crest": crest,
        "vib_clearance": clearance,
        "vib_shape": shape_factor,
        "vib_impulse": impulse_factor,
    }


# =============================================================================
# VIBRATION — FREQUENCY DOMAIN (Welch PSD)
# =============================================================================

def extract_vib_freq(x: np.ndarray) -> Dict[str, float]:
    """
    Extract frequency-domain vibration features from Welch PSD.
    Uses nperseg=4096 as specified; for W=8192 this averages 2 segments.
    
    Parameters
    ----------
    x : np.ndarray
        Zero-mean vibration window of length W.
        
    Returns
    -------
    dict
        Dictionary with exact column names as keys.
    """
    x = _validate_1d(x, W, "vib_freq")
    
    f, P = welch(x, fs=FS, nperseg=NPERSEG, nfft=NPERSEG, window="hann",
                 noverlap=0, return_onesided=True, scaling="density")
    
    p_0_1k   = _band_energy(P, f, 0.0, 1000.0)
    p_1_5k   = _band_energy(P, f, 1000.0, 5000.0)
    p_5_10k  = _band_energy(P, f, 5000.0, 10000.0)
    p_10_20k = _band_energy(P, f, 10000.0, 20000.0)
    p_20_32k = _band_energy(P, f, 20000.0, 32000.0)
    
    total_power = np.trapz(P, f)
    if total_power > 1e-20:
        centroid = np.trapz(f * P, f) / total_power
    else:
        centroid = 0.0
    
    p_norm = P / (np.sum(P) + 1e-20)
    p_norm = p_norm[p_norm > 1e-20]
    spectral_entropy = -np.sum(p_norm * np.log(p_norm))
    
    cumsum = np.cumsum(P)
    threshold = 0.85 * cumsum[-1]
    rolloff_idx = np.searchsorted(cumsum, threshold)
    rolloff = float(f[min(rolloff_idx, len(f) - 1)])
    
    mask_10k = f < 10000.0
    if np.any(mask_10k):
        dom_idx = np.argmax(P[mask_10k])
        dom_freq = float(f[mask_10k][dom_idx])
    else:
        dom_freq = 0.0
    
    ratio = p_5_10k / (p_0_1k + 1e-12)
    
    return {
        "vib_power_0_1k": float(p_0_1k),
        "vib_power_1_5k": float(p_1_5k),
        "vib_power_5_10k": float(p_5_10k),
        "vib_power_10_20k": float(p_10_20k),
        "vib_power_20_32k": float(p_20_32k),
        "vib_spectral_centroid": float(centroid),
        "vib_spectral_entropy": float(spectral_entropy),
        "vib_spectral_rolloff_85": rolloff,
        "vib_dominant_freq": dom_freq,
        "vib_band_ratio_5_10_0_1": float(ratio),
    }


# =============================================================================
# VIBRATION — ENVELOPE DEMODULATION
# =============================================================================

def extract_vib_envelope(x: np.ndarray) -> Dict[str, float]:
    """
    Extract envelope demodulation features from vibration.
    
    Bandpass [2000, 12000] Hz → Hilbert envelope → zero-mean → FFT.
    FFT is zero-padded to 2×W for finer frequency resolution.
    For W=8192, envelope FFT resolution = 3.906 Hz.
    
    Parameters
    ----------
    x : np.ndarray
        Zero-mean vibration window of length W.
        
    Returns
    -------
    dict
        Dictionary with exact column names as keys.
    """
    x = _validate_1d(x, W, "vib_envelope")
    
    sos = signal.butter(4, [2000.0, 12000.0], btype="bandpass", fs=FS, output="sos")
    filtered = signal.sosfiltfilt(sos, x)
    
    envelope = np.abs(hilbert(filtered))
    envelope = envelope - envelope.mean()
    
    n_fft = 2 * W
    env_fft = np.fft.rfft(envelope, n=n_fft)
    env_mag = np.abs(env_fft)
    env_freqs = np.fft.rfftfreq(n_fft, d=1.0 / FS)
    
    env_bpfo_1x = _sum_magnitude_near(env_mag, env_freqs, BPFO, 1.0)
    env_bpfo_2x = _sum_magnitude_near(env_mag, env_freqs, 2.0 * BPFO, 1.0)
    env_bpfo_3x = _sum_magnitude_near(env_mag, env_freqs, 3.0 * BPFO, 1.0)
    
    env_bpfi_1x = _sum_magnitude_near(env_mag, env_freqs, BPFI, 1.0)
    env_bpfi_2x = _sum_magnitude_near(env_mag, env_freqs, 2.0 * BPFI, 1.0)
    env_bpfi_3x = _sum_magnitude_near(env_mag, env_freqs, 3.0 * BPFI, 1.0)
    
    env_bpfi_sb_plus  = _sum_magnitude_near(env_mag, env_freqs, BPFI + FR, 1.0)
    env_bpfi_sb_minus = _sum_magnitude_near(env_mag, env_freqs, BPFI - FR, 1.0)
    env_bpfo_sb_plus  = _sum_magnitude_near(env_mag, env_freqs, BPFO + FR, 1.0)
    env_bpfo_sb_minus = _sum_magnitude_near(env_mag, env_freqs, BPFO - FR, 1.0)
    
    env_rms = float(np.sqrt(np.mean(envelope ** 2)))
    env_kurt = float(stats.kurtosis(envelope, fisher=True, bias=False))
    env_peak = float(np.max(np.abs(envelope)))
    
    return {
        "env_bpfo_1x": env_bpfo_1x,
        "env_bpfo_2x": env_bpfo_2x,
        "env_bpfo_3x": env_bpfo_3x,
        "env_bpfi_1x": env_bpfi_1x,
        "env_bpfi_2x": env_bpfi_2x,
        "env_bpfi_3x": env_bpfi_3x,
        "env_bpfi_sb_plus": env_bpfi_sb_plus,
        "env_bpfi_sb_minus": env_bpfi_sb_minus,
        "env_bpfo_sb_plus": env_bpfo_sb_plus,
        "env_bpfo_sb_minus": env_bpfo_sb_minus,
        "env_rms": env_rms,
        "env_kurt": env_kurt,
        "env_peak": env_peak,
    }


# =============================================================================
# CURRENT — TIME & FREQUENCY
# =============================================================================

def extract_current(x: np.ndarray, prefix: str) -> Dict[str, float]:
    """
    Extract time and frequency features for a single current channel.
    
    Parameters
    ----------
    x : np.ndarray
        Zero-mean current window of length W.
    prefix : str
        Either "i1" or "i2" for column naming.
        
    Returns
    -------
    dict
        Dictionary with exact column names as keys.
    """
    x = _validate_1d(x, W, f"current_{prefix}")
    
    rms_val = float(np.sqrt(np.mean(x ** 2)))
    std_val = float(x.std(ddof=0))
    peak_val = float(np.max(np.abs(x)))
    
    f, P = welch(x, fs=FS, nperseg=NPERSEG, nfft=NPERSEG, window="hann",
                 noverlap=0, return_onesided=True, scaling="density")
    
    p_50hz  = _band_energy(P, f, 50.0 - 2.0, 50.0 + 2.0)
    p_100hz = _band_energy(P, f, 100.0 - 2.0, 100.0 + 2.0)
    p_150hz = _band_energy(P, f, 150.0 - 2.0, 150.0 + 2.0)
    
    sb_fr_1x = (_sum_magnitude_near(P, f, FE + FR, 1.5) +
                _sum_magnitude_near(P, f, FE - FR, 1.5))
    sb_fr_2x = (_sum_magnitude_near(P, f, FE + 2.0 * FR, 1.5) +
                _sum_magnitude_near(P, f, FE - 2.0 * FR, 1.5))
    sb_bpfo = (_sum_magnitude_near(P, f, FE + BPFO, 1.5) +
               _sum_magnitude_near(P, f, FE - BPFO, 1.5))
    sb_bpfi = (_sum_magnitude_near(P, f, FE + BPFI, 1.5) +
               _sum_magnitude_near(P, f, FE - BPFI, 1.5))
    
    return {
        f"{prefix}_rms": rms_val,
        f"{prefix}_std": std_val,
        f"{prefix}_peak": peak_val,
        f"{prefix}_power_50hz": float(p_50hz),
        f"{prefix}_power_100hz": float(p_100hz),
        f"{prefix}_power_150hz": float(p_150hz),
        f"{prefix}_sideband_fr_1x": sb_fr_1x,
        f"{prefix}_sideband_fr_2x": sb_fr_2x,
        f"{prefix}_sideband_bpfo": sb_bpfo,
        f"{prefix}_sideband_bpfi": sb_bpfi,
    }


# =============================================================================
# CROSS-SENSOR COHERENCE
# =============================================================================

def extract_coherence(vib: np.ndarray, cur: np.ndarray) -> Dict[str, float]:
    """
    Compute mean magnitude-squared coherence between vibration and current
    in the 0–2000 Hz band.
    
    Parameters
    ----------
    vib : np.ndarray
        Zero-mean vibration window of length W.
    cur : np.ndarray
        Zero-mean current window of length W.
        
    Returns
    -------
    dict
        Dictionary with coherence feature.
    """
    vib = _validate_1d(vib, W, "coherence_vib")
    cur = _validate_1d(cur, W, "coherence_cur")
    
    f, C = coherence(vib, cur, fs=FS, nperseg=2048, window="hann")
    
    mask = (f >= 0.0) & (f <= 2000.0)
    if np.any(mask):
        mean_coh = float(np.mean(C[mask]))
    else:
        mean_coh = 0.0
    
    return {"vib_i1_coherence_mean_0_2k": mean_coh}


# =============================================================================
# MAIN PIPELINE
# =============================================================================

def extract_all_windows(df: pd.DataFrame, file_name: str) -> List[Dict]:
    """
    Extract features from all windows of a single file.
    
    Parameters
    ----------
    df : pd.DataFrame
        DataFrame with columns [phase_current_1, phase_current_2, vibration].
    file_name : str
        One of "HH", "IR", "OR".
        
    Returns
    -------
    List[Dict]
        List of feature dictionaries, one per window.
    """
    label = LABEL_MAP[file_name]
    vib_full = df["vibration"].values[:N]
    i1_full = df["phase_current_1"].values[:N]
    i2_full = df["phase_current_2"].values[:N]
    
    results = []
    n_windows = (N - W) // H + 1
    
    for w_id in range(n_windows):
        start = w_id * H
        end = start + W
        
        vib_win = vib_full[start:end].copy()
        i1_win = i1_full[start:end].copy()
        i2_win = i2_full[start:end].copy()
        
        vib_win -= vib_win.mean()
        i1_win -= i1_win.mean()
        i2_win -= i2_win.mean()
        
        feats = {}
        feats.update(extract_vib_time(vib_win))
        feats.update(extract_vib_freq(vib_win))
        feats.update(extract_vib_envelope(vib_win))
        feats.update(extract_current(i1_win, "i1"))
        feats.update(extract_current(i2_win, "i2"))
        feats.update(extract_coherence(vib_win, i1_win))
        
        feats["label"] = label
        feats["source_file"] = file_name
        feats["window_id"] = w_id
        
        results.append(feats)
    
    return results


def run_pipeline(base_path: Path) -> pd.DataFrame:
    """
    Run the complete feature extraction pipeline.
    
    Parameters
    ----------
    base_path : Path
        Directory containing HH.csv, IR.csv, OR.csv.
        
    Returns
    -------
    pd.DataFrame
        Feature matrix with all extracted features.
    """
    all_rows = []
    
    for name, fn in FILES.items():
        p = base_path / fn
        print(f"Loading {name} from {p} ...")
        df = pd.read_csv(p)
        print(f"  Raw shape: {df.shape}")
        
        rows = extract_all_windows(df, name)
        print(f"  Extracted {len(rows)} windows")
        all_rows.extend(rows)
    
    features_df = pd.DataFrame(all_rows)
    
    col_order = [
        "vib_mean", "vib_std", "vib_rms", "vib_peak", "vib_p2p",
        "vib_skew", "vib_kurt", "vib_crest", "vib_clearance",
        "vib_shape", "vib_impulse",
        "vib_power_0_1k", "vib_power_1_5k", "vib_power_5_10k",
        "vib_power_10_20k", "vib_power_20_32k",
        "vib_spectral_centroid", "vib_spectral_entropy",
        "vib_spectral_rolloff_85", "vib_dominant_freq",
        "vib_band_ratio_5_10_0_1",
        "env_bpfo_1x", "env_bpfo_2x", "env_bpfo_3x",
        "env_bpfi_1x", "env_bpfi_2x", "env_bpfi_3x",
        "env_bpfi_sb_plus", "env_bpfi_sb_minus",
        "env_bpfo_sb_plus", "env_bpfo_sb_minus",
        "env_rms", "env_kurt", "env_peak",
        "i1_rms", "i1_std", "i1_peak",
        "i1_power_50hz", "i1_power_100hz", "i1_power_150hz",
        "i1_sideband_fr_1x", "i1_sideband_fr_2x",
        "i1_sideband_bpfo", "i1_sideband_bpfi",
        "i2_rms", "i2_std", "i2_peak",
        "i2_power_50hz", "i2_power_100hz", "i2_power_150hz",
        "i2_sideband_fr_1x", "i2_sideband_fr_2x",
        "i2_sideband_bpfo", "i2_sideband_bpfi",
        "vib_i1_coherence_mean_0_2k",
        "label", "source_file", "window_id",
    ]
    
    available_cols = [c for c in col_order if c in features_df.columns]
    features_df = features_df[available_cols]
    
    return features_df


# =============================================================================
# VALIDATION REPORT
# =============================================================================

def print_validation_report(df: pd.DataFrame) -> None:
    """Print a comprehensive validation report."""
    print("\n" + "=" * 70)
    print("VALIDATION REPORT")
    print("=" * 70)
    
    print(f"\nTotal rows:    {len(df)}")
    print(f"Total columns: {len(df.columns)}")
    
    nan_counts = df.isna().sum()
    inf_counts = np.isinf(df.select_dtypes(include=[np.number])).sum()
    
    has_issues = False
    if nan_counts.sum() > 0:
        print("\nNaN counts per column:")
        for col, cnt in nan_counts.items():
            if cnt > 0:
                print(f"  {col}: {cnt}")
                has_issues = True
    else:
        print("\nNaN counts: NONE")
    
    if inf_counts.sum() > 0:
        print("\nInf counts per column:")
        for col, cnt in inf_counts.items():
            if cnt > 0:
                print(f"  {col}: {cnt}")
                has_issues = True
    else:
        print("Inf counts: NONE")
    
    if not has_issues:
        print("✓ All numeric columns are clean (no NaN, no Inf)")
    
    print("\n--- vib_rms per class ---")
    for cls, name in [(0, "HH"), (1, "IR"), (2, "OR")]:
        subset = df[df["label"] == cls]["vib_rms"]
        print(f"  {name}: mean={subset.mean():.6f}, std={subset.std():.6f}")
    
    print("\n--- Envelope Fault Confirmation ---")
    hh_bpfi = df[df["source_file"] == "HH"]["env_bpfi_1x"].mean()
    ir_bpfi = df[df["source_file"] == "IR"]["env_bpfi_1x"].mean()
    hh_bpfo = df[df["source_file"] == "HH"]["env_bpfo_1x"].mean()
    or_bpfo = df[df["source_file"] == "OR"]["env_bpfo_1x"].mean()
    
    print(f"  env_bpfi_1x  HH={hh_bpfi:.6f}  IR={ir_bpfi:.6f}  "
          f"IR > HH: {ir_bpfi > hh_bpfi}")
    print(f"  env_bpfo_1x  HH={hh_bpfo:.6f}  OR={or_bpfo:.6f}  "
          f"OR > HH: {or_bpfo > hh_bpfo}")
    
    if ir_bpfi > hh_bpfi and or_bpfo > hh_bpfo:
        print("  ✓ Fault signatures confirmed in envelope domain")
    else:
        print("  ⚠ Fault signature confirmation failed — inspect window size")
    
    print("\n" + "=" * 70)


# =============================================================================
# WINDOW SIZE COMPARISON (run this on your data to validate)
# =============================================================================

def compare_window_sizes(base_path: Path, candidates: List[int]) -> pd.DataFrame:
    """
    Compare candidate window sizes on actual data.
    Evaluates: sample count, kurtosis stability, and envelope peak prominence.
    
    Parameters
    ----------
    base_path : Path
        Directory containing CSV files.
    candidates : List[int]
        List of window sizes to test.
        
    Returns
    -------
    pd.DataFrame
        Comparison table.
    """
    global W, H
    
    rows = []
    for W_test in candidates:
        H_test = W_test // 2
        n_win = (N - W_test) // H_test + 1
        
        old_W, old_H = W, H
        W, H = W_test, H_test
        
        try:
            df_test = run_pipeline(base_path)
            
            kurt_stds = []
            for name in ["HH", "IR", "OR"]:
                subset = df_test[df_test["source_file"] == name]
                if len(subset) > 1:
                    kurt_stds.append(subset["vib_kurt"].std())
            avg_kurt_std = np.mean(kurt_stds) if kurt_stds else np.nan
            
            hh_bpfi = df_test[df_test["source_file"] == "HH"]["env_bpfi_1x"].mean()
            ir_bpfi = df_test[df_test["source_file"] == "IR"]["env_bpfi_1x"].mean()
            bpfi_ratio = ir_bpfi / (hh_bpfi + 1e-20)
            
            rows.append({
                "W": W_test,
                "duration_ms": W_test / FS * 1000,
                "n_windows": n_win,
                "total_rows": 3 * n_win,
                "welch_df_Hz": FS / NPERSEG,
                "env_fft_df_Hz": FS / (2 * W_test),
                "avg_kurt_std": round(avg_kurt_std, 4),
                "bpfi_IR_HH_ratio": round(bpfi_ratio, 2),
            })
        except Exception as e:
            rows.append({
                "W": W_test,
                "duration_ms": W_test / FS * 1000,
                "n_windows": n_win,
                "total_rows": 3 * n_win,
                "welch_df_Hz": FS / NPERSEG,
                "env_fft_df_Hz": FS / (2 * W_test),
                "avg_kurt_std": "ERROR",
                "bpfi_IR_HH_ratio": str(e),
            })
        finally:
            W, H = old_W, old_H
    
    return pd.DataFrame(rows)


# =============================================================================
# ENTRY POINT
# =============================================================================

if __name__ == "__main__":
    BASE = Path("/home/shawky/Documents/nti/actual nti/paderborn/"
                "Processed subset used in our STR-DDPM experiments/"
                "paderborn_subset_used_in_this_study/"
                "University of Paderborn Electric Motor Dataset/")
    
    print("=" * 70)
    print("PADERBORN BEARING FEATURE EXTRACTION PIPELINE")
    print("=" * 70)
    print(f"Window size:   {W} samples ({W/FS*1000:.1f} ms)")
    print(f"Hop size:      {H} samples ({H/FS*1000:.1f} ms)")
    print(f"Welch nperseg: {NPERSEG} samples (Δf = {FS/NPERSEG:.2f} Hz)")
    print(f"Env FFT Δf:    {FS/(2*W):.3f} Hz (2× zero-padded)")
    print(f"Windows/file:  {(N - W) // H + 1}")
    print(f"Expected rows: {3 * ((N - W) // H + 1)}")
    print("=" * 70)
    
    features_df = run_pipeline(BASE)
    
    out_path = Path("/home/shawky/Documents/nti/actual nti/paderborn/Processed subset used in our STR-DDPM experiments/paderborn_subset_used_in_this_study/University of Paderborn Electric Motor Dataset/paderborn_features.csv")
    out_path.parent.mkdir(parents=True, exist_ok=True)
    features_df.to_csv(out_path, index=False)
    print(f"\nSaved to: {out_path}")
    print(f"Shape: {features_df.shape}")
    
    print_validation_report(features_df)

PADERBORN BEARING FEATURE EXTRACTION PIPELINE
Window size:   8192 samples (128.0 ms)
Hop size:      4096 samples (64.0 ms)
Welch nperseg: 4096 samples (Δf = 15.62 Hz)
Env FFT Δf:    3.906 Hz (2× zero-padded)
Windows/file:  61
Expected rows: 183
Loading HH from /home/shawky/Documents/nti/actual nti/paderborn/Processed subset used in our STR-DDPM experiments/paderborn_subset_used_in_this_study/University of Paderborn Electric Motor Dataset/HH.csv ...
  Raw shape: (256001, 3)
  Extracted 61 windows
Loading IR from /home/shawky/Documents/nti/actual nti/paderborn/Processed subset used in our STR-DDPM experiments/paderborn_subset_used_in_this_study/University of Paderborn Electric Motor Dataset/IR.csv ...
  Raw shape: (258008, 3)
  Extracted 61 windows
Loading OR from /home/shawky/Documents/nti/actual nti/paderborn/Processed subset used in our STR-DDPM experiments/paderborn_subset_used_in_this_study/University of Paderborn Electric Motor Dataset/OR.csv ...
  Raw shape: (256000, 3)
  Extracte